In [1]:
import pandas as pd

In [2]:
kaggle1 = pd.read_csv("../data/raw_kaggle/customer_shopping_data.csv")
kaggle2 = pd.read_csv("../data/raw_kaggle/retail_sales_dataset.csv")

In [3]:
# Standardize Dataset 1
df1_clean = kaggle1[["customer_id", "gender", "age", "category", "quantity"]].copy()
df1_clean.columns = ["raw_user_id", "gender", "age", "category", "quantity"]

# Standardize Dataset 2
df2_clean = kaggle2[
    ["Customer ID", "Gender", "Age", "Product Category", "Quantity"]
].copy()
df2_clean.columns = ["raw_user_id", "gender", "age", "category", "quantity"]

In [4]:
# Merge into a single interactions dataframe
interactions_df = pd.concat([df1_clean, df2_clean], ignore_index=True)
# Drop duplicates (if a user bought the same category multiple times, we aggregate the quantity)
interactions_df = interactions_df.groupby(
    ["raw_user_id", "gender", "age", "category"], as_index=False
)["quantity"].sum()

In [5]:
interactions_df.head()

,raw_user_id,gender,age,category,quantity
0,C100004,Male,61,Clothing,5
1,C100005,Male,34,Shoes,2
2,C100006,Male,44,Toys,3
3,C100012,Male,25,Food & Beverage,5
4,C100019,Female,21,Toys,1


In [6]:
# Bridging Kaggle to MySQL (ID Mapping)

# 1. Convert the columns to 'category' dtype first
interactions_df["raw_user_id"] = interactions_df["raw_user_id"].astype("category")
interactions_df["category"] = interactions_df["category"].astype("category")

# 2. Generate your codes/IDs
interactions_df["user_id"] = interactions_df["raw_user_id"].cat.codes
interactions_df["item_id"] = interactions_df["category"].cat.codes

# 3. Create Mapping Dictionaries
user_id_map = dict(enumerate(interactions_df["raw_user_id"].cat.categories))
item_id_map = dict(enumerate(interactions_df["category"].cat.categories))

In [7]:
# Example: Save these maps so FastAPI knows that matrix index '0' = DB user 'C241288'
import json

with open("user_map.json", "w") as f:
    json.dump(user_id_map, f)
with open("item_map.json", "w") as f:
    json.dump(item_id_map, f)

In [8]:
# Constructing the Sparse Matrix
from scipy.sparse import coo_matrix

# Determine dimensions
num_users = interactions_df["user_id"].nunique()
num_items = interactions_df["item_id"].nunique()

# Create the Coordinate (COO) Sparse Matrix
interaction_matrix = coo_matrix(
    (
        interactions_df["quantity"],
        (interactions_df["user_id"], interactions_df["item_id"]),
    ),
    shape=(num_users, num_items),
)

In [11]:
# Training the Model (The Core Engine)
# LightFM is a high-performance Python library for hybrid recommendation systems, combining collaborative and content-based filtering to address sparse data and cold-start problems

from lightfm import LightFM

# # Initialize the model with WARP loss
# model = LightFM(loss="warp", no_components=30, learning_rate=0.05, random_state=42)

# # Train the model
# print("Training model...")
# model.fit(interaction_matrix, epochs=30, num_threads=4)
# print("Training complete.")

**model evaluation**

In [12]:
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, auc_score

In [13]:
# 1. Split the data (80% training, 20% testing)
# Assuming 'interaction_matrix' is already created from our previous step
train_matrix, test_matrix = random_train_test_split(
    interaction_matrix, test_percentage=0.2, random_state=42
)
# 2. Initialize and train the model on the TRAINING set only
model = LightFM(loss="warp", no_components=35, learning_rate=0.01, random_state=42)
print("Training model on the training set...")
model.fit(train_matrix, epochs=30, num_threads=4)

# 3. Evaluate: Precision@K
# We use k=5 to simulate showing the user 5 products in a "Recommended for You" carousel
print("Calculating Precision@K...")
train_precision = precision_at_k(model, train_matrix, k=5).mean()
test_precision = precision_at_k(
    model, test_matrix, train_interactions=train_matrix, k=5
).mean()

# 4. Evaluate: AUC (Area Under the ROC Curve)
print("Calculating AUC...")
train_auc = auc_score(model, train_matrix).mean()
test_auc = auc_score(model, test_matrix, train_interactions=train_matrix).mean()

# 5. Output the results
print(f"Train Precision@5: {train_precision:.4f}")
print(f"Test Precision@5:  {test_precision:.4f}")
print(f"Train AUC:         {train_auc:.4f}")
print(f"Test AUC:          {test_auc:.4f}")

Training model on the training set...
Calculating Precision@K...
Calculating AUC...
Train Precision@5: 0.1987
Test Precision@5:  0.1559
Train AUC:         0.9951
Test AUC:          0.7477


In [14]:
import pickle

# Export the trained model for the FastAPI service
with open("../models/saved/v1_retailiq_lightfm.pkl", "wb") as f:
    pickle.dump(model, f)